In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

print("Drive exists:", os.path.exists('/content/drive'))
print("Drive contents:", os.listdir('/content/drive'))

Drive exists: True
Drive contents: ['.shortcut-targets-by-id', 'MyDrive', '.Trash-0', '.Encrypted']


In [ ]:
import os

files = os.listdir('/content/drive/MyDrive')

for file in files:
    print(file)

Sheeja _April_Task Completion Report & Work Load Calculation - Template.xlsx
wpi-ierh-gng – 28 Apr 2021.pdf
kdn-bysj-cec – 20 Jun 2021.pdf
bdr-vftk-ots – 24 Jun 2021.pdf
Screenshot_2021-11-27-07-14-48-14 (1).jpg
IMG_20220511_134706_290.jpg
Soloop_20220508091638.mp4
KANNAPIES
VID20220502123319.mp4
Screenshot_2021-11-27-07-14-48-14.jpg
FAMILY PICS
TRASH
Untitled form (File responses)
Screenshot_2023-10-25-17-31-11-40_a63b0f8076346d26cbdc1b971a1da2a7.jpg
IMG_20230722_135944_0870.jpg
IMG_20230722_135947_0356.jpg
Gokul doc
cps homework-1.ipynb
Anokha
Pazham
Gokul credentials.gsheet
JPMR Mal.m4a
Colab Notebooks
Sub1
dl project
Untitled presentation.gslides
LearnSyncAI


In [ ]:
import os

project_path = '/content/drive/MyDrive/LearnSyncAI'

print("Project folders:")
print(os.listdir(project_path))

Project folders:
['Member_3', 'data', 'Member_1', 'Member_2']


In [ ]:
member3_path = os.path.join(project_path, 'Member_3')

print("Member 3 files:")
print(os.listdir(member3_path))

Member 3 files:
['member3_development.ipynb', 'schema.sql']


In [ ]:
schema_path = os.path.join(member3_path, "schema.sql")

with open(schema_path, "r") as file:
    schema_content = file.read()

print(schema_content)


-- ==========================================
-- LearnSync AI
-- Member 3: Database Schema
-- ==========================================

-- 1. STUDENTS TABLE
CREATE TABLE IF NOT EXISTS students (
    student_id VARCHAR(100) PRIMARY KEY,
    created_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);


-- 2. ATTEMPTS TABLE
CREATE TABLE IF NOT EXISTS attempts (
    attempt_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    student_id VARCHAR(100) NOT NULL,
    question_id VARCHAR(100) NOT NULL,
    concept_id VARCHAR(50) NOT NULL,
    classification VARCHAR(30),
    outcome INTEGER,
    evaluation JSONB,
    attempted_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,

    FOREIGN KEY (student_id)
        REFERENCES students(student_id),

    CHECK (outcome IN (0, 1) OR outcome IS NULL),

    CHECK (
        classification IN (
            'correct',
            'partially_correct',
            'incorrect'
        ) OR classification IS NULL
    )
);


-- 3. MASTERY TABLE
CREATE TABLE IF NOT EXIST

In [ ]:
for i, line in enumerate(schema_content.splitlines(), 1):
    print(f"{i}: {line}")

1: 
2: -- ==========================================
3: -- LearnSync AI
4: -- Member 3: Database Schema
5: -- ==========================================
6: 
7: -- 1. STUDENTS TABLE
8: CREATE TABLE IF NOT EXISTS students (
9:     student_id VARCHAR(100) PRIMARY KEY,
10:     created_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
11: );
12: 
13: 
14: -- 2. ATTEMPTS TABLE
15: CREATE TABLE IF NOT EXISTS attempts (
16:     attempt_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
17:     student_id VARCHAR(100) NOT NULL,
18:     question_id VARCHAR(100) NOT NULL,
19:     concept_id VARCHAR(50) NOT NULL,
20:     classification VARCHAR(30),
21:     outcome INTEGER,
22:     evaluation JSONB,
23:     attempted_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
24: 
25:     FOREIGN KEY (student_id)
26:         REFERENCES students(student_id),
27: 
28:     CHECK (outcome IN (0, 1) OR outcome IS NULL),
29: 
30:     CHECK (
31:         classification IN (
32:             'correct',
33:             'partially_cor

In [ ]:
import os

member3_path = '/content/drive/MyDrive/LearnSyncAI/Member_3'

print("Member 3 folder:", member3_path)
print("Files:", os.listdir(member3_path))

Member 3 folder: /content/drive/MyDrive/LearnSyncAI/Member_3
Files: ['member3_development.ipynb', 'schema.sql']


In [ ]:
!pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 14.6 MB/s  0:00:00


In [ ]:
%%writefile /content/drive/MyDrive/LearnSyncAI/Member_3/database.py

import os
import psycopg2
from psycopg2.extras import Json, RealDictCursor


# DATABASE CONNECTION
def get_connection():

    database_url = os.environ.get("DATABASE_URL")

    if not database_url:
        raise RuntimeError("DATABASE_URL is not configured")

    return psycopg2.connect(database_url)


# 1. CREATE STUDENT
def create_student(student_id):

    query = """
        INSERT INTO students (student_id)
        VALUES (%s)
        ON CONFLICT (student_id) DO NOTHING;
    """

    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(query, (student_id,))

    return True


# 2. SAVE ATTEMPT
def save_attempt(student_id, question_id,
                 concept_id, evaluation):

    classification = evaluation.get("classification")
    outcome = evaluation.get("outcome")

    if classification not in (
        None, "correct", "partially_correct", "incorrect"
    ):
        raise ValueError("Invalid classification")

    if outcome not in (None, 0, 1):
        raise ValueError("Outcome must be 0, 1 or None")

    query = """
        INSERT INTO attempts (
            student_id,
            question_id,
            concept_id,
            classification,
            outcome,
            evaluation
        )
        VALUES (%s, %s, %s, %s, %s, %s);
    """

    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(
                query,
                (
                    student_id,
                    question_id,
                    concept_id,
                    classification,
                    outcome,
                    Json(evaluation)
                )
            )

    return True


# 3. SAVE MASTERY
def save_mastery(student_id, mastery_profile):

    query = """
        INSERT INTO mastery (
            student_id,
            concept_id,
            mastery,
            updated_at
        )
        VALUES (%s, %s, %s, CURRENT_TIMESTAMP)
        ON CONFLICT (student_id, concept_id)
        DO UPDATE SET
            mastery = EXCLUDED.mastery,
            updated_at = CURRENT_TIMESTAMP;
    """

    with get_connection() as conn:
        with conn.cursor() as cur:

            for concept_id, mastery_score in mastery_profile.items():

                if not 0 <= mastery_score <= 1:
                    raise ValueError(
                        f"Invalid mastery score: {mastery_score}"
                    )

                cur.execute(
                    query,
                    (student_id, concept_id, mastery_score)
                )

    return True


# 4. GET STUDENT HISTORY
def get_student_history(student_id):

    query = """
        SELECT
            history_id,
            student_id,
            concept_id,
            activity,
            reason,
            next_action,
            created_at
        FROM learning_history
        WHERE student_id = %s
        ORDER BY created_at DESC;
    """

    with get_connection() as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:

            cur.execute(query, (student_id,))
            history = cur.fetchall()

    return [dict(row) for row in history]

Writing /content/drive/MyDrive/LearnSyncAI/Member_3/database.py


In [ ]:
print(os.listdir(member3_path))

['member3_development.ipynb', 'schema.sql', 'database.py']


In [ ]:
import py_compile

file_path = member3_path + "/database.py"

try:
    py_compile.compile(file_path, doraise=True)
    print("SUCCESS: No syntax errors in database.py")
except Exception as e:
    print("ERROR:", e)

SUCCESS: No syntax errors in database.py


In [ ]:
import sys

sys.path.append(member3_path)

import database

print("Available functions:")

print([
    name for name in dir(database)
    if not name.startswith("_")
])

Available functions:
['Json', 'RealDictCursor', 'create_student', 'get_connection', 'get_student_history', 'os', 'psycopg2', 'save_attempt', 'save_mastery']


In [ ]:
import inspect

functions = [
    database.create_student,
    database.save_attempt,
    database.save_mastery,
    database.get_student_history
]

for func in functions:
    print(func.__name__, inspect.signature(func))

create_student (student_id)
save_attempt (student_id, question_id, concept_id, evaluation)
save_mastery (student_id, mastery_profile)
get_student_history (student_id)


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import sys
import os

# Location of your Member 3 folder
member3_path = "/content/drive/MyDrive/LearnSyncAI/Member_3"

# Check whether database.py exists
print("Folder exists:", os.path.exists(member3_path))
print("Files:", os.listdir(member3_path))

# Add folder to Python's import path
if member3_path not in sys.path:
    sys.path.insert(0, member3_path)

import database

print("Database module imported successfully!")


Mounted at /content/drive
Folder exists: True
Files: ['schema.sql', 'database.py', '__pycache__', 'member3_development.ipynb']
Database module imported successfully!


In [ ]:

from google.colab import userdata
import os
import database

# Load the Neon connection string from Colab Secrets
os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")

# Connect to Neon
conn = database.get_connection()

cur = conn.cursor()
cur.execute("SELECT version();")

print("Neon connection successful!")
print("PostgreSQL version:", cur.fetchone()[0])

cur.close()
conn.close()


Neon connection successful!
PostgreSQL version: PostgreSQL 18.7 (a21afb5) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


In [ ]:
with open(f"{member3_path}/schema.sql", "r") as f:
    print(f.read())


-- ==========================================
-- LearnSync AI
-- Member 3: Database Schema
-- ==========================================

-- 1. STUDENTS TABLE
CREATE TABLE IF NOT EXISTS students (
    student_id VARCHAR(100) PRIMARY KEY,
    created_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);


-- 2. ATTEMPTS TABLE
CREATE TABLE IF NOT EXISTS attempts (
    attempt_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    student_id VARCHAR(100) NOT NULL,
    question_id VARCHAR(100) NOT NULL,
    concept_id VARCHAR(50) NOT NULL,
    classification VARCHAR(30),
    outcome INTEGER,
    evaluation JSONB,
    attempted_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,

    FOREIGN KEY (student_id)
        REFERENCES students(student_id),

    CHECK (outcome IN (0, 1) OR outcome IS NULL),

    CHECK (
        classification IN (
            'correct',
            'partially_correct',
            'incorrect'
        ) OR classification IS NULL
    )
);


-- 3. MASTERY TABLE
CREATE TABLE IF NOT EXIST

In [ ]:

import os
from google.colab import userdata
import psycopg2

# Get the Neon connection string securely
os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")

# Read your existing schema.sql file
schema_path = f"{member3_path}/schema.sql"

with open(schema_path, "r") as file:
    schema_sql = file.read()

# Connect to Neon and execute the schema
conn = psycopg2.connect(os.environ["DATABASE_URL"])
cur = conn.cursor()

try:
    cur.execute(schema_sql)
    conn.commit()
    print("All four tables created successfully!")

except Exception as e:
    conn.rollback()
    print("Error creating tables:", e)

finally:
    cur.close()
    conn.close()


All four tables created successfully!


In [ ]:
conn = psycopg2.connect(os.environ["DATABASE_URL"])
cur = conn.cursor()

cur.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    AND table_type = 'BASE TABLE'
    ORDER BY table_name;
""")

print("Tables in Neon:")
for row in cur.fetchall():
    print("-", row[0])

cur.close()
conn.close()

Tables in Neon:
- attempts
- learning_history
- mastery
- students


In [ ]:
import inspect
import database

print("Available database functions:")
print(inspect.signature(database.create_student))
print(inspect.signature(database.save_attempt))
print(inspect.signature(database.save_mastery))
print(inspect.signature(database.get_student_history))

Available database functions:
(student_id)
(student_id, question_id, concept_id, evaluation)
(student_id, mastery_profile)
(student_id)


In [ ]:
import database
import psycopg2
import os

# Create a sample test student
test_id = "TEST_MEMBER3_001"

database.create_student(test_id)

# Verify the student exists in Neon
conn = psycopg2.connect(os.environ["DATABASE_URL"])
cur = conn.cursor()

cur.execute(
    "SELECT student_id FROM students WHERE student_id = %s",
    (test_id,)
)

result = cur.fetchone()

if result:
    print("SUCCESS! Student created in Neon:", result[0])
else:
    print("Student was not found.")

cur.close()
conn.close()

SUCCESS! Student created in Neon: TEST_MEMBER3_001


In [ ]:
import database

test_evaluation = {
    "classification": "correct",
    "outcome": 1,
    "feedback": "Test attempt successful"
}

database.save_attempt(
    student_id="TEST_MEMBER3_001",
    question_id="TEST_Q001",
    concept_id="TEST_C001",
    evaluation=test_evaluation
)

print("Attempt saved successfully!")

Attempt saved successfully!


In [ ]:
database.save_mastery(
    student_id="TEST_MEMBER3_001",
    mastery_profile={
        "TEST_C001": 0.85,
        "TEST_C002": 0.60
    }
)

print("Mastery scores saved successfully!")

Mastery scores saved successfully!


In [ ]:
history = database.get_student_history("TEST_MEMBER3_001")

print("Learning history:")
print(history)

Learning history:
[]


In [ ]:
import psycopg2
import os

conn = psycopg2.connect(os.environ["DATABASE_URL"])
cur = conn.cursor()

cur.execute("""
    INSERT INTO learning_history
    (student_id, concept_id, activity, reason, next_action)
    VALUES (%s, %s, %s, %s, %s)
""", (
    "TEST_MEMBER3_001",
    "TEST_C001",
    "revision",
    "Test history record",
    "practice"
))

conn.commit()
cur.close()
conn.close()

print("Test learning history inserted!")

Test learning history inserted!


In [ ]:
history = database.get_student_history("TEST_MEMBER3_001")
print(history)

[{'history_id': 1, 'student_id': 'TEST_MEMBER3_001', 'concept_id': 'TEST_C001', 'activity': 'revision', 'reason': 'Test history record', 'next_action': 'practice', 'created_at': datetime.datetime(2026, 11, 15, 17, 41, 51, 911602, tzinfo=datetime.timezone.utc)}]


In [ ]:
with open(f"{member3_path}/database.py", "r") as file:
    print(file.read())


import os
import psycopg2
from psycopg2.extras import Json, RealDictCursor


# DATABASE CONNECTION
def get_connection():

    database_url = os.environ.get("DATABASE_URL")

    if not database_url:
        raise RuntimeError("DATABASE_URL is not configured")

    return psycopg2.connect(database_url)


# 1. CREATE STUDENT
def create_student(student_id):

    query = """
        INSERT INTO students (student_id)
        VALUES (%s)
        ON CONFLICT (student_id) DO NOTHING;
    """

    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(query, (student_id,))

    return True


# 2. SAVE ATTEMPT
def save_attempt(student_id, question_id,
                 concept_id, evaluation):

    classification = evaluation.get("classification")
    outcome = evaluation.get("outcome")

    if classification not in (
        None, "correct", "partially_correct", "incorrect"
    ):
        raise ValueError("Invalid classification")

    if outcome not in (None, 

In [ ]:
import shutil
import os

database_path = f"{member3_path}/database.py"
backup_path = f"{member3_path}/database_backup.py"

shutil.copy2(database_path, backup_path)

print("Backup created successfully!")
print("Location:", backup_path)

Backup created successfully!
Location: /content/drive/MyDrive/LearnSyncAI/Member_3/database_backup.py


In [ ]:
import sys
import importlib
import inspect
import py_compile

database_path = f"{member3_path}/database.py"

# Check syntax
py_compile.compile(database_path, doraise=True)

# Reload updated module
if "database" in sys.modules:
    importlib.reload(sys.modules["database"])
else:
    import database

import database

print("Compilation successful!")
print("Module imported successfully!")

print(inspect.signature(database.create_student))
print(inspect.signature(database.save_attempt))
print(inspect.signature(database.save_mastery))
print(inspect.signature(database.get_student_history))

Compilation successful!
Module imported successfully!
(student_id)
(student_id, question_id, concept_id, evaluation)
(student_id, mastery_profile)
(student_id)


In [ ]:
# Test 1: Invalid student ID
try:
    database.create_student("")
except ValueError as e:
    print("Test 1 PASSED:", e)


# Test 2: Invalid mastery score
try:
    database.save_mastery(
        "TEST_MEMBER3_001",
        {"arrays": 1.5}
    )
except ValueError as e:
    print("Test 2 PASSED:", e)


# Test 3: Invalid attempt outcome
try:
    database.save_attempt(
        "TEST_MEMBER3_001",
        "TEST_Q002",
        "arrays",
        {
            "classification": "correct",
            "outcome": 5
        }
    )
except ValueError as e:
    print("Test 3 PASSED:", e)

Test 1 PASSED: student_id must be a non-empty string.
Test 2 PASSED: Invalid mastery score for arrays: 1.5. Expected a number from 0 to 1.
Test 3 PASSED: outcome must be 0, 1 or None.


In [ ]:
# 1. Create student (safe to run again)
database.create_student("TEST_MEMBER3_001")
print("1. Student creation: PASSED")

# 2. Save another attempt
database.save_attempt(
    "TEST_MEMBER3_001",
    "TEST_Q003",
    "arrays",
    {
        "classification": "correct",
        "outcome": 1,
        "feedback": "Improved database test"
    }
)
print("2. Attempt saving: PASSED")

# 3. Save mastery
database.save_mastery(
    "TEST_MEMBER3_001",
    {
        "arrays": 0.90,
        "linked_lists": 0.70
    }
)
print("3. Mastery saving: PASSED")

# 4. Retrieve history
history = database.get_student_history("TEST_MEMBER3_001")
print("4. History retrieval: PASSED")
print("History records:", len(history))

1. Student creation: PASSED
2. Attempt saving: PASSED
3. Mastery saving: PASSED
4. History retrieval: PASSED
History records: 1


In [ ]:
import os

curriculum_path = f"{member3_path}/curriculum.py"

print("File exists:", os.path.exists(curriculum_path))
print("File location:", curriculum_path)

File exists: False
File location: /content/drive/MyDrive/LearnSyncAI/Member_3/curriculum.py


In [ ]:
import os

os.makedirs(member3_path, exist_ok=True)

curriculum_path = os.path.join(member3_path, "curriculum.py")

print("Ready to create:", curriculum_path)

Ready to create: /content/drive/MyDrive/LearnSyncAI/Member_3/curriculum.py


In [43]:

%%writefile /content/drive/MyDrive/LearnSyncAI/Member_3/curriculum.py

"""
LearnSyncAI - Curriculum Planning Module
Member 3

Selects the next learning activity using:
1. Student mastery from the BKT Knowledge Tracing Agent
2. Strict prerequisite relationships
3. Recommended learning relationships
4. Previous learning history
"""

# All agreed DSA concept IDs
DSA_CONCEPTS = [
    "arrays",
    "linked_lists",
    "stacks",
    "queues",
    "recursion",
    "trees",
    "binary_search_trees",
    "heaps",
    "sorting",
    "hashing",
    "graphs",
    "bfs",
    "dfs"
]

# Strict prerequisites:
# A concept is held back until its strict prerequisites
# reach the configured mastery threshold.
STRICT_PREREQUISITES = {
    "arrays": [],
    "linked_lists": [],
    "stacks": [],
    "queues": [],
    "recursion": [],
    "trees": [],
    "binary_search_trees": ["trees"],
    "heaps": ["trees"],
    "sorting": [],
    "hashing": [],
    "graphs": [],
    "bfs": ["graphs"],
    "dfs": ["graphs"]
}

# Recommended prerequisites:
# These guide learning order but NEVER block a student.
RECOMMENDED_PREREQUISITES = {
    "arrays": [],
    "linked_lists": ["arrays"],
    "stacks": [],
    "queues": [],
    "recursion": [],
    "trees": [],
    "binary_search_trees": [],
    "heaps": [],
    "sorting": ["arrays"],
    "hashing": [],
    "graphs": [],
    "bfs": [],
    "dfs": []
}

# Configurable curriculum settings
STRICT_PREREQUISITE_THRESHOLD = 0.60

REVISION_THRESHOLD = 0.40
PRACTICE_THRESHOLD = 0.70
ADVANCE_THRESHOLD = 0.85

# Default prerequisite configuration
DSA_PREREQUISITES = {
    "strict": STRICT_PREREQUISITES,
    "recommended": RECOMMENDED_PREREQUISITES
}


def _extract_mastery(mastery_profile):
    """
    Accepts either:
    {"student_id": "...", "mastery": {...}}
    or a direct concept-to-mastery dictionary.
    """

    if not isinstance(mastery_profile, dict):
        raise TypeError("mastery_profile must be a dictionary")

    mastery = mastery_profile.get("mastery", mastery_profile)

    if not isinstance(mastery, dict):
        raise TypeError("Mastery values must be provided as a dictionary")

    return mastery


def _get_strictly_eligible_concepts(mastery, strict_prerequisites):
    """
    Returns concepts whose strict prerequisites
    have reached the required mastery threshold.
    """

    eligible = []

    for concept in DSA_CONCEPTS:
        prerequisites = strict_prerequisites.get(concept, [])

        if all(
            mastery.get(prerequisite, 0.0)
            >= STRICT_PREREQUISITE_THRESHOLD
            for prerequisite in prerequisites
        ):
            eligible.append(concept)

    return eligible


def select_next_activity(
    student_id,
    mastery_profile,
    prerequisites,
    learning_history
):
    """
    Selects the next learning activity for a student.

    Required interface:
    select_next_activity(
        student_id,
        mastery_profile,
        prerequisites,
        learning_history
    )

    Returns a structured curriculum decision.
    """

    if not isinstance(student_id, str) or not student_id.strip():
        raise ValueError("student_id must be a non-empty string")

    mastery = _extract_mastery(mastery_profile)

    # Use the default configuration if none is supplied.
    if prerequisites is None:
        prerequisites = DSA_PREREQUISITES

    if not isinstance(prerequisites, dict):
        raise TypeError("prerequisites must be a dictionary")

    # Support both the new configurable structure and
    # a simple prerequisite dictionary for compatibility.
    if "strict" in prerequisites or "recommended" in prerequisites:
        strict = prerequisites.get("strict", {})
        recommended = prerequisites.get("recommended", {})
    else:
        strict = prerequisites
        recommended = RECOMMENDED_PREREQUISITES

    if not isinstance(strict, dict) or not isinstance(recommended, dict):
        raise TypeError("Prerequisite configurations must be dictionaries")

    if not isinstance(learning_history, list):
        raise TypeError("learning_history must be a list")

    # Find concepts that satisfy strict prerequisites.
    eligible = _get_strictly_eligible_concepts(mastery, strict)

    if not eligible:
        raise ValueError("No concepts currently satisfy strict prerequisites")

   # Prioritize eligible concepts that have never been assessed.
unassessed = [
    concept for concept in eligible
    if concept not in mastery
]

if unassessed:
    selected_concept = unassessed[0]
    current_mastery = None
else:
    # Select the eligible concept with the lowest mastery.
    selected_concept = min(
        eligible,
        key=lambda concept: mastery[concept]
    )
    current_mastery = mastery[selected_concept]

    # Decide the activity using mastery thresholds.
    if current_mastery is None:
        activity = "reassessment"
        reason = "Concept has not been assessed yet"
        next_action = "reassess_mastery"

    elif current_mastery < REVISION_THRESHOLD:
        activity = "revision"
        reason = "Mastery below configured revision threshold"
        next_action = "retrieve_content"

    elif current_mastery < PRACTICE_THRESHOLD:
        activity = "practice"
        reason = "Student needs additional practice"
        next_action = "generate_question"

    elif current_mastery < ADVANCE_THRESHOLD:
        activity = "reassessment"
        reason = "Student is ready for a mastery reassessment"
        next_action = "reassess_mastery"

    else:
        activity = "advance"
        reason = "Mastery reached the configured advancement threshold"
        next_action = "select_next_concept"

    # Recommended prerequisites influence the explanation,
    # but never prevent concept selection.
    recommended_for_concept = recommended.get(selected_concept, [])

    unmet_recommended = [
        concept for concept in recommended_for_concept
        if mastery.get(concept, 0.0) < STRICT_PREREQUISITE_THRESHOLD
    ]

    if unmet_recommended:
        reason += (
            ". Recommended prior concepts with lower mastery: "
            + ", ".join(unmet_recommended)
            + ". Learning is not blocked."
        )

    # Check the most recent activity for this concept.
    previous_activities = [
        item for item in learning_history
        if item.get("concept_id") == selected_concept
    ]

    if previous_activities:
        reason += ". Previous learning activity found in history"

    return {
        "student_id": student_id,
        "concept_id": selected_concept,
        "activity": activity,
        "reason": reason,
        "next_action": next_action
    }


Overwriting /content/drive/MyDrive/LearnSyncAI/Member_3/curriculum.py


In [ ]:
import os

curriculum_path = os.path.join(member3_path, "curriculum.py")

print("File created:", os.path.exists(curriculum_path))
print("File size:", os.path.getsize(curriculum_path), "bytes")

File created: True
File size: 6230 bytes


In [ ]:
import sys

if member3_path not in sys.path:
    sys.path.insert(0, member3_path)

import curriculum

print("1. Curriculum module imported: PASSED")

# Test 1: Recommended prerequisite should NOT block Linked Lists
mastery_profile = {
    "student_id": "TEST_MEMBER3_001",
    "mastery": {
        "arrays": 0.20,
        "linked_lists": 0.10,
        "stacks": 0.90,
        "queues": 0.90,
        "recursion": 0.90,
        "trees": 0.90,
        "binary_search_trees": 0.90,
        "heaps": 0.90,
        "sorting": 0.90,
        "hashing": 0.90,
        "graphs": 0.90,
        "bfs": 0.90,
        "dfs": 0.90
    }
}

result = curriculum.select_next_activity(
    student_id="TEST_MEMBER3_001",
    mastery_profile=mastery_profile,
    prerequisites=curriculum.DSA_PREREQUISITES,
    learning_history=[]
)

print("\nTest 1 result:")
print(result)

assert result["concept_id"] == "linked_lists", (
    "Linked Lists should remain eligible despite low Arrays mastery"
)

print("\n2. Recommended prerequisite test: PASSED")

# Test 2: BST must be blocked when Trees mastery is low
mastery_profile["mastery"]["trees"] = 0.20

eligible = curriculum._get_strictly_eligible_concepts(
    mastery_profile["mastery"],
    curriculum.STRICT_PREREQUISITES
)

assert "binary_search_trees" not in eligible
assert "heaps" not in eligible

print("3. Strict prerequisite test: PASSED")

# Test 3: BST becomes eligible when Trees mastery reaches 0.60
mastery_profile["mastery"]["trees"] = 0.60

eligible = curriculum._get_strictly_eligible_concepts(
    mastery_profile["mastery"],
    curriculum.STRICT_PREREQUISITES
)

assert "binary_search_trees" in eligible
assert "heaps" in eligible

print("4. Strict prerequisite mastery threshold test: PASSED")

1. Curriculum module imported: PASSED

Test 1 result:
{'student_id': 'TEST_MEMBER3_001', 'concept_id': 'linked_lists', 'activity': 'revision', 'reason': 'Mastery below configured revision threshold. Recommended prior concepts with lower mastery: arrays. Learning is not blocked.', 'next_action': 'retrieve_content'}

2. Recommended prerequisite test: PASSED
3. Strict prerequisite test: PASSED
4. Strict prerequisite mastery threshold test: PASSED


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

member3_path = "/content/drive/MyDrive/LearnSyncAI/Member_3"

print("Folder exists:", os.path.exists(member3_path))
print("Files inside Member 3:")

for file in os.listdir(member3_path):
    print(file)

Folder exists: True
Files inside Member 3:
member3_development.ipynb
schema.sql
database_backup.py
__pycache__
database.py
curriculum.py


In [ ]:
import sys
import os

if member3_path not in sys.path:
    sys.path.insert(0, member3_path)

import database
import curriculum

print("Database module imported successfully!")
print("Curriculum module imported successfully!")

Database module imported successfully!
Curriculum module imported successfully!


In [ ]:
print("DATABASE FUNCTIONS:")
for name in dir(database):
    if not name.startswith("_") and callable(getattr(database, name)):
        print("-", name)

print("\nCURRICULUM FUNCTIONS:")
for name in dir(curriculum):
    if not name.startswith("_") and callable(getattr(curriculum, name)):
        print("-", name)

DATABASE FUNCTIONS:
- Json
- Real
- RealDictCursor
- contextmanager
- create_student
- get_connection
- get_student_history
- save_attempt
- save_mastery

CURRICULUM FUNCTIONS:
- select_next_activity


In [ ]:
import inspect

print("DATABASE FUNCTION INPUTS:")

for name in [
    "create_student",
    "save_attempt",
    "save_mastery",
    "get_student_history"
]:
    function = getattr(database, name)
    print(name, inspect.signature(function))

print("\nCURRICULUM FUNCTION INPUTS:")
print(inspect.signature(curriculum.select_next_activity))

DATABASE FUNCTION INPUTS:
create_student (student_id)
save_attempt (student_id, question_id, concept_id, evaluation)
save_mastery (student_id, mastery_profile)
get_student_history (student_id)

CURRICULUM FUNCTION INPUTS:
(student_id, mastery_profile, prerequisites, learning_history)


In [ ]:

%%writefile /content/drive/MyDrive/LearnSyncAI/Member_3/orchestrator.py

import database
import curriculum


def run_learning_cycle(student_id, mastery_profile, prerequisites=None):
    """
    Runs the Member 3 learning cycle.

    1. Registers the student.
    2. Retrieves the student's learning history.
    3. Uses the curriculum planner to select the next activity.
    4. Returns the learning decision.

    The mastery profile is supplied by the caller until
    a database retrieval function is implemented.
    """

    if not student_id:
        raise ValueError("student_id is required")

    if not isinstance(mastery_profile, dict):
        raise TypeError("mastery_profile must be a dictionary")

    # Step 1: Ensure the student exists
    database.create_student(student_id)

    # Step 2: Retrieve learning history
    learning_history = database.get_student_history(student_id)

    # Step 3: Use the existing prerequisite configuration
    if prerequisites is None:
        prerequisites = curriculum.DSA_PREREQUISITES

    # Step 4: Select the next learning activity
    decision = curriculum.select_next_activity(
        student_id,
        mastery_profile,
        prerequisites,
        learning_history
    )

    # Step 5: Return the result
    return {
        "student_id": student_id,
        "learning_history": learning_history,
        "decision": decision
    }


Writing /content/drive/MyDrive/LearnSyncAI/Member_3/orchestrator.py


In [ ]:
import importlib
import orchestrator

importlib.reload(orchestrator)

print("Orchestrator imported successfully!")
print("Available function:", orchestrator.run_learning_cycle)

Orchestrator imported successfully!
Available function: <function run_learning_cycle at 0x7bdc5246fd80>


In [ ]:
import os

print("DATABASE_URL configured:", bool(os.environ.get("DATABASE_URL")))

DATABASE_URL configured: False


In [ ]:
import os
from google.colab import userdata

os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")

print("DATABASE_URL configured:", bool(os.environ.get("DATABASE_URL")))

DATABASE_URL configured: True


In [ ]:
try:
    conn = database.get_connection()
    print("Connected to Neon PostgreSQL successfully!")
    conn.close()
except Exception as e:
    print("Database connection failed:")
    print(type(e).__name__, str(e))

Connected to Neon PostgreSQL successfully!


In [ ]:
test_student_id = "phase1_test_student"

test_mastery = {
    "arrays": 0.90,
    "linked_lists": 0.30,
    "stacks": 0.50,
    "queues": 0.50,
    "recursion": 0.50,
    "trees": 0.20,
    "binary_search_trees": 0.10,
    "heaps": 0.10,
    "sorting": 0.60,
    "hashing": 0.50,
    "graphs": 0.20,
    "bfs": 0.10,
    "dfs": 0.10
}

result = orchestrator.run_learning_cycle(
    test_student_id,
    test_mastery
)

print("Learning cycle completed!")
print(result)

Learning cycle completed!
{'student_id': 'phase1_test_student', 'learning_history': [], 'decision': {'student_id': 'phase1_test_student', 'concept_id': 'trees', 'activity': 'revision', 'reason': 'Mastery below configured revision threshold', 'next_action': 'retrieve_content'}}


In [ ]:

%%writefile /content/drive/MyDrive/LearnSyncAI/Member_3/mock_agents.py

"""
Temporary mock agents for LearnSyncAI Phase 1.

These simulate Member 1 and Member 2 modules.
They will be replaced with the actual team modules during Phase 2.
"""


def mock_generate_question(concept_id):
    """Simulates Member 2's RAG question generation."""

    return {
        "question_id": f"mock_{concept_id}_001",
        "concept_id": concept_id,
        "question": f"Explain the main idea of {concept_id}."
    }


def mock_evaluate_answer(answer):
    """Simulates Member 2's answer evaluator."""

    if not isinstance(answer, str) or not answer.strip():
        return {
            "classification": "incorrect",
            "outcome": 0,
            "feedback": "Please provide an answer."
        }

    return {
        "classification": "correct",
        "outcome": 1,
        "feedback": "Mock evaluation: answer accepted."
    }


def mock_update_mastery(old_mastery, concept_id, outcome):
    """Simulates Member 1's BKT mastery update."""

    updated_mastery = dict(old_mastery)

    current_score = updated_mastery.get(concept_id, 0.0)

    if outcome == 1:
        updated_mastery[concept_id] = min(
            1.0, current_score + 0.10
        )
    else:
        updated_mastery[concept_id] = max(
            0.0, current_score - 0.05
        )

    return updated_mastery


Writing /content/drive/MyDrive/LearnSyncAI/Member_3/mock_agents.py


In [ ]:
import importlib
import mock_agents

importlib.reload(mock_agents)

print("Mock agents imported successfully!")

# Test question generation
question = mock_agents.mock_generate_question("trees")
print("\nGenerated question:")
print(question)

# Test answer evaluation
evaluation = mock_agents.mock_evaluate_answer(
    "A tree is a hierarchical data structure."
)
print("\nEvaluation:")
print(evaluation)

# Test mastery update
updated = mock_agents.mock_update_mastery(
    {"trees": 0.20},
    "trees",
    evaluation["outcome"]
)
print("\nUpdated mastery:")
print(updated)

Mock agents imported successfully!

Generated question:
{'question_id': 'mock_trees_001', 'concept_id': 'trees', 'question': 'Explain the main idea of trees.'}

Evaluation:
{'classification': 'correct', 'outcome': 1, 'feedback': 'Mock evaluation: answer accepted.'}

Updated mastery:
{'trees': 0.30000000000000004}


In [ ]:
import inspect

print(inspect.getsource(orchestrator.run_learning_cycle))

def run_learning_cycle(student_id, mastery_profile, prerequisites=None):
    """
    Runs the Member 3 learning cycle.

    1. Registers the student.
    2. Retrieves the student's learning history.
    3. Uses the curriculum planner to select the next activity.
    4. Returns the learning decision.
    
    The mastery profile is supplied by the caller until
    a database retrieval function is implemented.
    """

    if not student_id:
        raise ValueError("student_id is required")

    if not isinstance(mastery_profile, dict):
        raise TypeError("mastery_profile must be a dictionary")

    # Step 1: Ensure the student exists
    database.create_student(student_id)

    # Step 2: Retrieve learning history
    learning_history = database.get_student_history(student_id)

    # Step 3: Use the existing prerequisite configuration
    if prerequisites is None:
        prerequisites = curriculum.DSA_PREREQUISITES

    # Step 4: Select the next learning activity
    decision = cu

In [ ]:

%%writefile /content/drive/MyDrive/LearnSyncAI/Member_3/orchestrator.py

import database
import curriculum
import mock_agents


def run_learning_cycle(student_id, mastery_profile, prerequisites=None):
    """Select the next learning activity for a student."""

    if not student_id:
        raise ValueError("student_id is required")

    if not isinstance(mastery_profile, dict):
        raise TypeError("mastery_profile must be a dictionary")

    database.create_student(student_id)

    learning_history = database.get_student_history(student_id)

    if prerequisites is None:
        prerequisites = curriculum.DSA_PREREQUISITES

    decision = curriculum.select_next_activity(
        student_id,
        mastery_profile,
        prerequisites,
        learning_history
    )

    return {
        "student_id": student_id,
        "learning_history": learning_history,
        "decision": decision
    }


def get_next_question(student_id, mastery_profile, prerequisites=None):
    """Select a learning activity and generate a question."""

    cycle = run_learning_cycle(
        student_id,
        mastery_profile,
        prerequisites
    )

    concept_id = cycle["decision"]["concept_id"]

    question = mock_agents.mock_generate_question(concept_id)

    return {
        "student_id": student_id,
        "decision": cycle["decision"],
        "question": question
    }


def process_student_answer(student_id, question, answer, mastery_profile):
    """Evaluate an answer and save the attempt and updated mastery."""

    evaluation = mock_agents.mock_evaluate_answer(answer)

    concept_id = question["concept_id"]
    question_id = question["question_id"]

    updated_mastery = mock_agents.mock_update_mastery(
        mastery_profile,
        concept_id,
        evaluation["outcome"]
    )

    database.save_attempt(
        student_id,
        question_id,
        concept_id,
        evaluation
    )

    database.save_mastery(
        student_id,
        updated_mastery
    )

    return {
        "student_id": student_id,
        "question": question,
        "evaluation": evaluation,
        "updated_mastery": updated_mastery
    }


Overwriting /content/drive/MyDrive/LearnSyncAI/Member_3/orchestrator.py


In [ ]:
import importlib
import mock_agents
import orchestrator

importlib.reload(mock_agents)
importlib.reload(orchestrator)

print("Updated orchestrator loaded successfully!")

print("Available functions:")
print("- run_learning_cycle:", callable(orchestrator.run_learning_cycle))
print("- get_next_question:", callable(orchestrator.get_next_question))
print("- process_student_answer:", callable(orchestrator.process_student_answer))

Updated orchestrator loaded successfully!
Available functions:
- run_learning_cycle: True
- get_next_question: True
- process_student_answer: True


In [ ]:
test_student_id = "phase1_test_student"

question_result = orchestrator.get_next_question(
    test_student_id,
    test_mastery
)

print("Selected learning activity:")
print(question_result["decision"])

print("\nGenerated question:")
print(question_result["question"])

Selected learning activity:
{'student_id': 'phase1_test_student', 'concept_id': 'trees', 'activity': 'revision', 'reason': 'Mastery below configured revision threshold', 'next_action': 'retrieve_content'}

Generated question:
{'question_id': 'mock_trees_001', 'concept_id': 'trees', 'question': 'Explain the main idea of trees.'}


In [ ]:
student_answer = "A tree is a hierarchical data structure made up of nodes connected by edges."

answer_result = orchestrator.process_student_answer(
    test_student_id,
    question_result["question"],
    student_answer,
    test_mastery
)

print("Answer processing completed!")

print("\nEvaluation:")
print(answer_result["evaluation"])

print("\nUpdated mastery:")
print(answer_result["updated_mastery"])

Answer processing completed!

Evaluation:
{'classification': 'correct', 'outcome': 1, 'feedback': 'Mock evaluation: answer accepted.'}

Updated mastery:
{'arrays': 0.9, 'linked_lists': 0.3, 'stacks': 0.5, 'queues': 0.5, 'recursion': 0.5, 'trees': 0.30000000000000004, 'binary_search_trees': 0.1, 'heaps': 0.1, 'sorting': 0.6, 'hashing': 0.5, 'graphs': 0.2, 'bfs': 0.1, 'dfs': 0.1}


In [ ]:
history = database.get_student_history(test_student_id)

print("Number of learning history records:", len(history))

print("\nAttempt saved successfully if no database error occurred.")

Number of learning history records: 0

Attempt saved successfully if no database error occurred.


In [ ]:
conn = database.get_connection()
cur = conn.cursor()

cur.execute("""
    SELECT question_id, concept_id, classification, outcome
    FROM attempts
    WHERE student_id = %s
""", (test_student_id,))

print("SAVED ATTEMPTS:")
for row in cur.fetchall():
    print(row)

cur.execute("""
    SELECT concept_id, mastery
    FROM mastery
    WHERE student_id = %s
""", (test_student_id,))

print("\nSAVED MASTERY:")
for row in cur.fetchall():
    print(row)

cur.close()
conn.close()

SAVED ATTEMPTS:
('mock_trees_001', 'trees', 'correct', 1)

SAVED MASTERY:
('arrays', 0.9)
('bfs', 0.1)
('binary_search_trees', 0.1)
('dfs', 0.1)
('graphs', 0.2)
('hashing', 0.5)
('heaps', 0.1)
('linked_lists', 0.3)
('queues', 0.5)
('recursion', 0.5)
('sorting', 0.6)
('stacks', 0.5)
('trees', 0.30000000000000004)


In [ ]:
import inspect

print(inspect.getsource(database.get_student_history))
print("\nExisting database functions inspected.")

def get_student_history(student_id):
    """
    Retrieve a student's learning history.

    Args:
        student_id (str): Student identifier.

    Returns:
        list[dict]: Learning history records ordered
                    from newest to oldest.

        Returns an empty list if no records exist.

    Raises:
        ValueError: If student_id is invalid.
        psycopg2.Error: If the database query fails.
    """

    _validate_required_string(student_id, "student_id")

    query = """
        SELECT
            history_id,
            student_id,
            concept_id,
            activity,
            reason,
            next_action,
            created_at

        FROM learning_history

        WHERE student_id = %s

        ORDER BY created_at DESC;
    """

    with _managed_connection() as conn:
        with conn.cursor(
            cursor_factory=RealDictCursor
        ) as cur:

            cur.execute(query, (student_id,))
            history = cur.fetchall()

    r

In [ ]:

new_function = '''

def save_learning_history(student_id, concept_id, activity, reason, next_action):
    """
    Save a student's curriculum decision to learning_history.
    """

    _validate_required_string(student_id, "student_id")
    _validate_required_string(concept_id, "concept_id")
    _validate_required_string(activity, "activity")
    _validate_required_string(reason, "reason")
    _validate_required_string(next_action, "next_action")

    allowed_activities = {
        "revision",
        "practice",
        "reassessment",
        "advance"
    }

    if activity not in allowed_activities:
        raise ValueError("Invalid learning activity")

    query = """
        INSERT INTO learning_history (
            student_id,
            concept_id,
            activity,
            reason,
            next_action
        )
        VALUES (%s, %s, %s, %s, %s)
        RETURNING history_id;
    """

    with _managed_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(
                query,
                (student_id, concept_id, activity, reason, next_action)
            )
            history_id = cur.fetchone()[0]

    return history_id

'''

database_file = os.path.join(member3_path, "database.py")

with open(database_file, "a") as file:
    file.write(new_function)

print("save_learning_history() added to database.py")


save_learning_history() added to database.py


In [ ]:
import importlib

importlib.reload(database)

print("New function available:",
      callable(database.save_learning_history))

print("\nFunction details:")
print(inspect.signature(database.save_learning_history))

New function available: True

Function details:
(student_id, concept_id, activity, reason, next_action)


In [ ]:
history_id = database.save_learning_history(
    student_id=test_student_id,
    concept_id="trees",
    activity="revision",
    reason="Mastery is below the revision threshold.",
    next_action="Review tree concepts and examples."
)

print("Learning history saved!")
print("History ID:", history_id)

Learning history saved!
History ID: 2


In [ ]:
print("LEARNING CYCLE FUNCTION:")
print(inspect.getsource(orchestrator.run_learning_cycle))

print("\nQUESTION SELECTION FUNCTION:")
print(inspect.getsource(orchestrator.get_next_question))

LEARNING CYCLE FUNCTION:
def run_learning_cycle(student_id, mastery_profile, prerequisites=None):
    """Select the next learning activity for a student."""

    if not student_id:
        raise ValueError("student_id is required")

    if not isinstance(mastery_profile, dict):
        raise TypeError("mastery_profile must be a dictionary")

    database.create_student(student_id)

    learning_history = database.get_student_history(student_id)

    if prerequisites is None:
        prerequisites = curriculum.DSA_PREREQUISITES

    decision = curriculum.select_next_activity(
        student_id,
        mastery_profile,
        prerequisites,
        learning_history
    )

    return {
        "student_id": student_id,
        "learning_history": learning_history,
        "decision": decision
    }


QUESTION SELECTION FUNCTION:
def get_next_question(student_id, mastery_profile, prerequisites=None):
    """Select a learning activity and generate a question."""

    cycle = run_learning

In [ ]:

orchestrator_file = os.path.join(member3_path, "orchestrator.py")

with open(orchestrator_file, "r") as file:
    source = file.read()

marker = "# AUTOMATIC_LEARNING_HISTORY_V1"

if marker not in source:
    updated_function = '''

# AUTOMATIC_LEARNING_HISTORY_V1
def run_learning_cycle(student_id, mastery_profile, prerequisites=None):
    """Select the next activity and save the decision to learning history."""

    if not student_id:
        raise ValueError("student_id is required")

    if not isinstance(mastery_profile, dict):
        raise TypeError("mastery_profile must be a dictionary")

    # 1. Ensure student exists
    database.create_student(student_id)

    # 2. Retrieve previous learning history
    learning_history = database.get_student_history(student_id)

    # 3. Load prerequisite configuration
    if prerequisites is None:
        prerequisites = curriculum.DSA_PREREQUISITES

    # 4. Select the next activity
    decision = curriculum.select_next_activity(
        student_id,
        mastery_profile,
        prerequisites,
        learning_history
    )

    # 5. Save the new learning decision
    history_id = database.save_learning_history(
        student_id=student_id,
        concept_id=decision["concept_id"],
        activity=decision["activity"],
        reason=decision["reason"],
        next_action=decision["next_action"]
    )

    # 6. Retrieve updated history
    updated_history = database.get_student_history(student_id)

    return {
        "student_id": student_id,
        "learning_history": updated_history,
        "decision": decision,
        "history_id": history_id
    }
'''

    with open(orchestrator_file, "a") as file:
        file.write(updated_function)

    print("Orchestrator updated successfully!")

else:
    print("Automatic history saving is already added.")


Orchestrator updated successfully!


In [ ]:
import importlib

# Reload database first, then orchestrator
importlib.reload(database)
importlib.reload(orchestrator)

print("Database reloaded.")
print("Orchestrator reloaded.")

# Run a fresh test using a separate student
auto_test_id = "phase1_auto_history_test"

result = orchestrator.run_learning_cycle(
    auto_test_id,
    test_mastery
)

print("\nSelected decision:")
print(result["decision"])

print("\nSaved history ID:")
print(result["history_id"])

print("\nHistory records:")
print(len(result["learning_history"]))

Database reloaded.
Orchestrator reloaded.

Selected decision:
{'student_id': 'phase1_auto_history_test', 'concept_id': 'trees', 'activity': 'revision', 'reason': 'Mastery below configured revision threshold', 'next_action': 'retrieve_content'}

Saved history ID:
3

History records:
1


In [ ]:
question_test = orchestrator.get_next_question(
    auto_test_id,
    test_mastery
)

print("Selected decision:")
print(question_test["decision"])

print("\nGenerated question:")
print(question_test["question"])

Selected decision:
{'student_id': 'phase1_auto_history_test', 'concept_id': 'trees', 'activity': 'revision', 'reason': 'Mastery below configured revision threshold. Previous learning activity found in history', 'next_action': 'retrieve_content'}

Generated question:
{'question_id': 'mock_trees_001', 'concept_id': 'trees', 'question': 'Explain the main idea of trees.'}


In [ ]:
student_answer = (
    "A tree is a hierarchical data structure "
    "made up of nodes connected by edges."
)

answer_test = orchestrator.process_student_answer(
    auto_test_id,
    question_test["question"],
    student_answer,
    test_mastery
)

print("Evaluation:")
print(answer_test["evaluation"])

print("\nUpdated mastery:")
print(answer_test["updated_mastery"])

Evaluation:
{'classification': 'correct', 'outcome': 1, 'feedback': 'Mock evaluation: answer accepted.'}

Updated mastery:
{'arrays': 0.9, 'linked_lists': 0.3, 'stacks': 0.5, 'queues': 0.5, 'recursion': 0.5, 'trees': 0.30000000000000004, 'binary_search_trees': 0.1, 'heaps': 0.1, 'sorting': 0.6, 'hashing': 0.5, 'graphs': 0.2, 'bfs': 0.1, 'dfs': 0.1}


In [ ]:
conn = database.get_connection()
cur = conn.cursor()

cur.execute("""
    SELECT question_id, concept_id, classification, outcome
    FROM attempts
    WHERE student_id = %s
    ORDER BY attempted_at DESC
    LIMIT 5;
""", (auto_test_id,))

print("SAVED ATTEMPTS:")
for row in cur.fetchall():
    print(row)

cur.execute("""
    SELECT concept_id, mastery
    FROM mastery
    WHERE student_id = %s
    ORDER BY concept_id;
""", (auto_test_id,))

print("\nSAVED MASTERY:")
for row in cur.fetchall():
    print(row)

cur.close()
conn.close()

SAVED ATTEMPTS:
('mock_trees_001', 'trees', 'correct', 1)

SAVED MASTERY:
('arrays', 0.9)
('bfs', 0.1)
('binary_search_trees', 0.1)
('dfs', 0.1)
('graphs', 0.2)
('hashing', 0.5)
('heaps', 0.1)
('linked_lists', 0.3)
('queues', 0.5)
('recursion', 0.5)
('sorting', 0.6)
('stacks', 0.5)
('trees', 0.30000000000000004)


In [ ]:
import os

member3_path = "/content/drive/MyDrive/LearnSyncAI/Member_3"

print("Files in Member 3 folder:\n")

for filename in sorted(os.listdir(member3_path)):
    print(filename)

Files in Member 3 folder:

__pycache__
curriculum.py
database.py
database_backup.py
member3_development.ipynb
mock_agents.py
orchestrator.py
schema.sql


In [ ]:
import os

print("Checking Member 1 files:")

for filename in ["assessment.py", "bkt_engine.py"]:
    path = os.path.join(member3_path, filename)
    print(filename, "FOUND" if os.path.exists(path) else "NOT FOUND")

Checking Member 1 files:
assessment.py NOT FOUND
bkt_engine.py NOT FOUND


In [ ]:
from google.colab import files
import shutil
import os

uploaded = files.upload()

for filename in ["assessment.py", "bkt_engine.py"]:
    if filename in uploaded:
        shutil.move(
            filename,
            os.path.join(member3_path, filename)
        )
        print(filename, "copied successfully")
    else:
        print(filename, "was not uploaded")

Saving bkt_engine.py to bkt_engine.py
Saving assessment.py to assessment.py
assessment.py copied successfully
bkt_engine.py copied successfully


In [ ]:
import os

for filename in ["assessment.py", "bkt_engine.py"]:
    path = os.path.join(member3_path, filename)
    print(filename, "FOUND" if os.path.exists(path) else "NOT FOUND")

assessment.py FOUND
bkt_engine.py FOUND


In [ ]:
import sys

if member3_path not in sys.path:
    sys.path.insert(0, member3_path)

from assessment import get_assessment_questions, evaluate_assessment_answer
from bkt_engine import update_bkt, get_mastery_profile

print("Member 1 modules imported successfully!")

print("\nAssessment questions available:",
      len(get_assessment_questions()))

print("\nInitial mastery profile:")
print(get_mastery_profile("test_student"))

Member 1 modules imported successfully!

Assessment questions available: 13

Initial mastery profile:
{'student_id': 'test_student', 'mastery': {}}


In [ ]:
result = update_bkt(
    student_id="test_student",
    concept_id="arrays",
    outcome=1
)

print("BKT update result:")
print(result)

print("\nUpdated mastery profile:")
print(get_mastery_profile("test_student"))

BKT update result:
{'student_id': 'test_student', 'concept_id': 'arrays', 'outcome': 1, 'prior_mastery': 0.2, 'predicted_correct_probability': 0.3400000000000001, 'posterior_mastery': 0.5294117647058822, 'mastery': 0.5764705882352941}

Updated mastery profile:
{'student_id': 'test_student', 'mastery': {'arrays': 0.5764705882352941}}


In [ ]:
from google.colab import userdata
import os

os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")

print("DATABASE_URL configured:", bool(os.environ.get("DATABASE_URL")))

DATABASE_URL configured: True


In [ ]:
from database import create_student, save_mastery

student_id = "test_student"

create_student(student_id)

mastery_profile = get_mastery_profile(student_id)

save_mastery(student_id, mastery_profile)

print("Mastery saved successfully!")
print(mastery_profile)

ValueError: Invalid mastery score for student_id: test_student. Expected a number from 0 to 1.

In [ ]:
student_id = "test_student"

# Get the BKT profile
mastery_profile = get_mastery_profile(student_id)

# Extract only the concept mastery dictionary
mastery_data = mastery_profile["mastery"]

# Save it to PostgreSQL
save_mastery(student_id, mastery_data)

print("Mastery saved successfully!")
print(mastery_data)

Mastery saved successfully!
{'arrays': 0.5764705882352941}


In [ ]:
from database import get_connection
from psycopg2.extras import RealDictCursor

with get_connection() as conn:
    with conn.cursor(cursor_factory=RealDictCursor) as cur:
        cur.execute(
            """
            SELECT student_id, concept_id, mastery
            FROM mastery
            WHERE student_id = %s
            """,
            ("test_student",)
        )
        rows = cur.fetchall()

print("Mastery retrieved from Neon:")
for row in rows:
    print(dict(row))

Mastery retrieved from Neon:
{'student_id': 'test_student', 'concept_id': 'arrays', 'mastery': 0.5764705882352941}


In [ ]:
student_id = "test_student"

# Get one assessment question
questions = get_assessment_questions("arrays")
question = questions[0]

print("Question:")
print(question["question"])
print("\nOptions:")
for option in question["options"]:
    print(option)

Question:
What is the time complexity of accessing an element using its index in an array?

Options:
A
B
C
D


In [ ]:
print(question["options"])

{'A': 'O(1)', 'B': 'O(n)', 'C': 'O(log n)', 'D': 'O(n^2)'}


In [ ]:
evaluation = evaluate_assessment_answer(
    question["question_id"],
    "A"
)

print("Assessment evaluation:")
print(evaluation)

Assessment evaluation:
{'question_id': 'assess_arrays_01', 'concept_id': 'arrays', 'outcome': 1, 'classification': 'correct', 'feedback': 'Correct! Array elements can be accessed directly using their index.'}


In [ ]:
from database import save_attempt

print("save_attempt imported successfully!")

save_attempt imported successfully!


In [ ]:
save_attempt(
    student_id,
    evaluation["question_id"],
    evaluation["concept_id"],
    {
        **evaluation,
        "bkt_outcome": evaluation["outcome"]
    }
)

print("Assessment attempt saved successfully!")

Assessment attempt saved successfully!


In [ ]:
with get_connection() as conn:
    with conn.cursor(cursor_factory=RealDictCursor) as cur:
        cur.execute(
            """
            SELECT student_id, question_id, concept_id,
                   classification, outcome
            FROM attempts
            WHERE student_id = %s
            ORDER BY attempted_at DESC
            LIMIT 5
            """,
            ("test_student",)
        )
        attempts = cur.fetchall()

print("Saved attempts:")
for attempt in attempts:
    print(dict(attempt))

Saved attempts:
{'student_id': 'test_student', 'question_id': 'assess_arrays_01', 'concept_id': 'arrays', 'classification': 'correct', 'outcome': 1}


In [ ]:
from curriculum import select_next_activity, DSA_PREREQUISITES
from database import get_student_history

# Get the latest mastery from BKT
mastery_profile = get_mastery_profile(student_id)

# Get the student's learning history
learning_history = get_student_history(student_id)

# Ask curriculum planner to select the next activity
next_activity = select_next_activity(
    student_id,
    mastery_profile,
    DSA_PREREQUISITES,
    learning_history
)

print("Next learning activity:")
print(next_activity)

Next learning activity:
{'student_id': 'test_student', 'concept_id': 'linked_lists', 'activity': 'revision', 'reason': 'Mastery below configured revision threshold', 'next_action': 'retrieve_content'}


In [ ]:
from database import save_learning_history

save_learning_history(
    student_id,
    next_activity["concept_id"],
    next_activity["activity"],
    next_activity["reason"],
    next_activity["next_action"]
)

print("Curriculum activity saved successfully!")

Curriculum activity saved successfully!


In [ ]:
import ast
import os

orchestrator_path = os.path.join(
    member3_path, "orchestrator.py"
)

with open(orchestrator_path, "r") as file:
    source = file.read()

tree = ast.parse(source)

print("Functions in orchestrator.py:\n")

for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        print(node.name)

Functions in orchestrator.py:

run_learning_cycle
get_next_question
process_student_answer
run_learning_cycle


In [ ]:
with open(orchestrator_path, "r") as file:
    print(file.read())


import database
import curriculum
import mock_agents


def run_learning_cycle(student_id, mastery_profile, prerequisites=None):
    """Select the next learning activity for a student."""

    if not student_id:
        raise ValueError("student_id is required")

    if not isinstance(mastery_profile, dict):
        raise TypeError("mastery_profile must be a dictionary")

    database.create_student(student_id)

    learning_history = database.get_student_history(student_id)

    if prerequisites is None:
        prerequisites = curriculum.DSA_PREREQUISITES

    decision = curriculum.select_next_activity(
        student_id,
        mastery_profile,
        prerequisites,
        learning_history
    )

    return {
        "student_id": student_id,
        "learning_history": learning_history,
        "decision": decision
    }


def get_next_question(student_id, mastery_profile, prerequisites=None):
    """Select a learning activity and generate a question."""

    cycle = run_learni

In [ ]:
mock_agents_path = os.path.join(member3_path, "mock_agents.py")

with open(mock_agents_path, "r") as file:
    print(file.read())


"""
Temporary mock agents for LearnSyncAI Phase 1.

These simulate Member 1 and Member 2 modules.
They will be replaced with the actual team modules during Phase 2.
"""


def mock_generate_question(concept_id):
    """Simulates Member 2's RAG question generation."""

    return {
        "question_id": f"mock_{concept_id}_001",
        "concept_id": concept_id,
        "question": f"Explain the main idea of {concept_id}."
    }


def mock_evaluate_answer(answer):
    """Simulates Member 2's answer evaluator."""

    if not isinstance(answer, str) or not answer.strip():
        return {
            "classification": "incorrect",
            "outcome": 0,
            "feedback": "Please provide an answer."
        }

    return {
        "classification": "correct",
        "outcome": 1,
        "feedback": "Mock evaluation: answer accepted."
    }


def mock_update_mastery(old_mastery, concept_id, outcome):
    """Simulates Member 1's BKT mastery update."""

    updated_mastery = dict(

In [ ]:
from database import create_student, save_attempt, save_mastery
from assessment import evaluate_assessment_answer
from bkt_engine import update_bkt, get_mastery_profile


def process_real_assessment_answer(student_id, question, answer):

    # 1. Ensure student exists
    create_student(student_id)

    # 2. Evaluate using Member 1's real assessment module
    evaluation = evaluate_assessment_answer(
        question["question_id"],
        answer
    )

    # 3. Update using Member 1's real BKT engine
    bkt_result = update_bkt(
        student_id,
        evaluation["concept_id"],
        evaluation["outcome"]
    )

    # 4. Get updated mastery
    mastery_profile = get_mastery_profile(student_id)

    # 5. Save mastery in PostgreSQL
    save_mastery(
        student_id,
        mastery_profile["mastery"]
    )

    # 6. Save assessment attempt in PostgreSQL
    save_attempt(
        student_id,
        evaluation["question_id"],
        evaluation["concept_id"],
        {
            **evaluation,
            "bkt_outcome": evaluation["outcome"]
        }
    )

    return {
        "student_id": student_id,
        "evaluation": evaluation,
        "bkt_result": bkt_result,
        "mastery_profile": mastery_profile
    }


print("Real assessment-to-BKT integration function created!")

Real assessment-to-BKT integration function created!


In [ ]:
questions = get_assessment_questions("linked_lists")

test_question = questions[0]

print("Question ID:", test_question["question_id"])
print("\nQuestion:", test_question["question"])
print("\nOptions:")

for key, value in test_question["options"].items():
    print(key, ":", value)

Question ID: assess_linked_lists_01

Question: In a singly linked list, what does each node normally contain?

Options:
A : Only an array index
B : Data and a reference to the next node
C : Two child nodes
D : A priority value only


In [ ]:
evaluation = evaluate_assessment_answer(
    test_question["question_id"],
    "B"
)

print("Assessment result:")
print(evaluation)

Assessment result:
{'question_id': 'assess_linked_lists_01', 'concept_id': 'linked_lists', 'outcome': 1, 'classification': 'correct', 'feedback': 'Correct! A singly linked-list node stores data and a link to the next node.'}


In [ ]:
from bkt_engine import update_bkt, get_mastery_profile
from database import save_attempt, save_mastery

# 1. Update BKT mastery
bkt_result = update_bkt(
    student_id,
    evaluation["concept_id"],
    evaluation["outcome"]
)

# 2. Get updated mastery profile
mastery_profile = get_mastery_profile(student_id)

# 3. Save the attempt in PostgreSQL
save_attempt(
    student_id,
    evaluation["question_id"],
    evaluation["concept_id"],
    {
        **evaluation,
        "bkt_outcome": evaluation["outcome"]
    }
)

# 4. Save mastery in PostgreSQL
save_mastery(
    student_id,
    mastery_profile["mastery"]
)

print("BKT update:", bkt_result)
print("Updated mastery:", mastery_profile)
print("Attempt and mastery saved successfully!")

BKT update: {'student_id': 'test_student', 'concept_id': 'linked_lists', 'outcome': 1, 'prior_mastery': 0.2, 'predicted_correct_probability': 0.3400000000000001, 'posterior_mastery': 0.5294117647058822, 'mastery': 0.5764705882352941}
Updated mastery: {'student_id': 'test_student', 'mastery': {'arrays': 0.8736842105263158, 'linked_lists': 0.5764705882352941}}
Attempt and mastery saved successfully!


In [ ]:
from database import get_student_history, get_connection
from psycopg2.extras import RealDictCursor

print("Student learning history:")
print(get_student_history(student_id))

print("\nMastery retrieved from Neon:")

with get_connection() as conn:
    with conn.cursor(cursor_factory=RealDictCursor) as cur:
        cur.execute(
            """
            SELECT concept_id, mastery
            FROM mastery
            WHERE student_id = %s
            ORDER BY concept_id
            """,
            (student_id,)
        )
        for row in cur.fetchall():
            print(dict(row))

Student learning history:
[{'history_id': 5, 'student_id': 'test_student', 'concept_id': 'linked_lists', 'activity': 'revision', 'reason': 'Mastery below configured revision threshold', 'next_action': 'retrieve_content', 'created_at': datetime.datetime(2026, 11, 18, 10, 57, 24, 83799, tzinfo=datetime.timezone.utc)}]

Mastery retrieved from Neon:
{'concept_id': 'arrays', 'mastery': 0.8736842105263158}
{'concept_id': 'linked_lists', 'mastery': 0.5764705882352941}


In [ ]:
from curriculum import select_next_activity, DSA_PREREQUISITES
from database import get_student_history

# Retrieve mastery from PostgreSQL
mastery_data = {}

with get_connection() as conn:
    with conn.cursor(cursor_factory=RealDictCursor) as cur:
        cur.execute(
            """
            SELECT concept_id, mastery
            FROM mastery
            WHERE student_id = %s
            """,
            (student_id,)
        )

        for row in cur.fetchall():
            mastery_data[row["concept_id"]] = row["mastery"]

# Retrieve learning history
history = get_student_history(student_id)

# Select next activity
decision = select_next_activity(
    student_id,
    mastery_data,
    DSA_PREREQUISITES,
    history
)

print("Next curriculum decision:")
print(decision)

Next curriculum decision:
{'student_id': 'test_student', 'concept_id': 'stacks', 'activity': 'revision', 'reason': 'Mastery below configured revision threshold', 'next_action': 'retrieve_content'}


In [ ]:
from curriculum import select_next_activity, DSA_PREREQUISITES

test_mastery = {
    "arrays": 0.87,
    "linked_lists": 0.57
}

test_decision = select_next_activity(
    "test_student",
    test_mastery,
    DSA_PREREQUISITES,
    []
)

print(test_decision)

{'student_id': 'test_student', 'concept_id': 'stacks', 'activity': 'revision', 'reason': 'Mastery below configured revision threshold', 'next_action': 'retrieve_content'}


In [39]:
import importlib
import curriculum

importlib.reload(curriculum)

from curriculum import select_next_activity, DSA_PREREQUISITES

print("Updated curriculum module loaded!")

SyntaxError: 'return' outside function (curriculum.py, line 241)

In [40]:
file_path = "/content/drive/MyDrive/LearnSyncAI/Member_3/curriculum.py"

with open(file_path, "r") as file:
    lines = file.readlines()

for i in range(220, min(255, len(lines))):
    print(f"{i+1}: {lines[i]}", end="")

221:         concept for concept in recommended_for_concept
222:         if mastery.get(concept, 0.0) < STRICT_PREREQUISITE_THRESHOLD
223:     ]
224: 
225:     if unmet_recommended:
226:         reason += (
227:             ". Recommended prior concepts with lower mastery: "
228:             + ", ".join(unmet_recommended)
229:             + ". Learning is not blocked."
230:         )
231: 
232:     # Check the most recent activity for this concept.
233:     previous_activities = [
234:         item for item in learning_history
235:         if item.get("concept_id") == selected_concept
236:     ]
237: 
238:     if previous_activities:
239:         reason += ". Previous learning activity found in history"
240: 
241:     return {
242:         "student_id": student_id,
243:         "concept_id": selected_concept,
244:         "activity": activity,
245:         "reason": reason,
246:         "next_action": next_action
247:     }


In [41]:
file_path = "/content/drive/MyDrive/LearnSyncAI/Member_3/curriculum.py"

with open(file_path, "r") as file:
    lines = file.readlines()

print("Total lines:", len(lines))
print("\nLast 50 lines of curriculum.py:\n")

start = max(0, len(lines) - 50)

for i in range(start, len(lines)):
    print(f"{i+1}: {lines[i]}", end="")

Total lines: 247

Last 50 lines of curriculum.py:

198:     reason = "Mastery below configured revision threshold"
199:     next_action = "retrieve_content"
200: 
201: elif current_mastery < PRACTICE_THRESHOLD:
202:     activity = "practice"
203:     reason = "Student needs additional practice"
204:     next_action = "generate_question"
205: 
206: elif current_mastery < ADVANCE_THRESHOLD:
207:     activity = "reassessment"
208:     reason = "Student is ready for a mastery reassessment"
209:     next_action = "reassess_mastery"
210: 
211: else:
212:     activity = "advance"
213:     reason = "Mastery reached the configured advancement threshold"
214:     next_action = "select_next_concept"
215: 
216:     # Recommended prerequisites influence the explanation,
217:     # but never prevent concept selection.
218:     recommended_for_concept = recommended.get(selected_concept, [])
219: 
220:     unmet_recommended = [
221:         concept for concept in recommended_for_concept
222:         i

In [42]:
with open(file_path, "r") as file:
    lines = file.readlines()

for i in range(180, 220):
    print(f"{i+1}: {lines[i]}", end="")

181:     current_mastery = None
182: else:
183:     # Select the eligible concept with the lowest mastery.
184:     selected_concept = min(
185:         eligible,
186:         key=lambda concept: mastery[concept]
187:     )
188:     current_mastery = mastery[selected_concept]
189: 
190:    # Decide the activity using mastery thresholds.
191: if current_mastery is None:
192:     activity = "reassessment"
193:     reason = "Concept has not been assessed yet"
194:     next_action = "reassess_mastery"
195: 
196: elif current_mastery < REVISION_THRESHOLD:
197:     activity = "revision"
198:     reason = "Mastery below configured revision threshold"
199:     next_action = "retrieve_content"
200: 
201: elif current_mastery < PRACTICE_THRESHOLD:
202:     activity = "practice"
203:     reason = "Student needs additional practice"
204:     next_action = "generate_question"
205: 
206: elif current_mastery < ADVANCE_THRESHOLD:
207:     activity = "reassessment"
208:     reason = "Student is ready 

In [44]:
import sys
import importlib

member3_path = "/content/drive/MyDrive/LearnSyncAI/Member_3"

if member3_path not in sys.path:
    sys.path.insert(0, member3_path)

importlib.invalidate_caches()

if "curriculum" in sys.modules:
    del sys.modules["curriculum"]

import curriculum

print("Curriculum loaded successfully!")
print("Loaded from:", curriculum.__file__)

SyntaxError: 'return' outside function (curriculum.py, line 241)

In [45]:
file_path = "/content/drive/MyDrive/LearnSyncAI/Member_3/curriculum.py"

with open(file_path, "r") as file:
    lines = file.readlines()

for i in range(155, 200):
    print(f"{i+1}: {repr(lines[i])}")

156: '        recommended = prerequisites.get("recommended", {})\n'
157: '    else:\n'
158: '        strict = prerequisites\n'
159: '        recommended = RECOMMENDED_PREREQUISITES\n'
160: '\n'
161: '    if not isinstance(strict, dict) or not isinstance(recommended, dict):\n'
162: '        raise TypeError("Prerequisite configurations must be dictionaries")\n'
163: '\n'
164: '    if not isinstance(learning_history, list):\n'
165: '        raise TypeError("learning_history must be a list")\n'
166: '\n'
167: '    # Find concepts that satisfy strict prerequisites.\n'
168: '    eligible = _get_strictly_eligible_concepts(mastery, strict)\n'
169: '\n'
170: '    if not eligible:\n'
171: '        raise ValueError("No concepts currently satisfy strict prerequisites")\n'
172: '\n'
173: '   # Prioritize eligible concepts that have never been assessed.\n'
174: 'unassessed = [\n'
175: '    concept for concept in eligible\n'
176: '    if concept not in mastery\n'
177: ']\n'
178: '\n'
179: 'if unasses

In [46]:
file_path = "/content/drive/MyDrive/LearnSyncAI/Member_3/curriculum.py"

with open(file_path, "r") as file:
    content = file.read()

marker = "   # Prioritize eligible concepts that have never been assessed."

if marker not in content:
    raise ValueError("Could not find the expected section. No changes made.")

corrected_section = '''    # Prioritize eligible concepts that have never been assessed.
    unassessed = [
        concept for concept in eligible
        if concept not in mastery
    ]

    if unassessed:
        selected_concept = unassessed[0]
        current_mastery = None
    else:
        # Select the eligible concept with the lowest mastery.
        selected_concept = min(
            eligible,
            key=lambda concept: mastery[concept]
        )
        current_mastery = mastery[selected_concept]

    # Decide the activity using mastery thresholds.
    if current_mastery is None:
        activity = "reassessment"
        reason = "Concept has not been assessed yet"
        next_action = "reassess_mastery"

    elif current_mastery < REVISION_THRESHOLD:
        activity = "revision"
        reason = "Mastery below configured revision threshold"
        next_action = "retrieve_content"

    elif current_mastery < PRACTICE_THRESHOLD:
        activity = "practice"
        reason = "Student needs additional practice"
        next_action = "generate_question"

    elif current_mastery < ADVANCE_THRESHOLD:
        activity = "reassessment"
        reason = "Student is ready for a mastery reassessment"
        next_action = "reassess_mastery"

    else:
        activity = "advance"
        reason = "Mastery reached the configured advancement threshold"
        next_action = "select_next_concept"

    # Recommended prerequisites influence the explanation.
    recommended_for_concept = recommended.get(selected_concept, [])

    unmet_recommended = [
        concept for concept in recommended_for_concept
        if mastery.get(concept, 0.0) < STRICT_PREREQUISITE_THRESHOLD
    ]

    if unmet_recommended:
        reason += (
            ". Recommended prior concepts with lower mastery: "
            + ", ".join(unmet_recommended)
            + ". Learning is not blocked."
        )

    # Check previous learning activity.
    previous_activities = [
        item for item in learning_history
        if item.get("concept_id") == selected_concept
    ]

    if previous_activities:
        reason += ". Previous learning activity found in history"

    return {
        "student_id": student_id,
        "concept_id": selected_concept,
        "activity": activity,
        "reason": reason,
        "next_action": next_action
    }
'''

content = content[:content.index(marker)] + corrected_section

with open(file_path, "w") as file:
    file.write(content)

print("Curriculum indentation corrected and file saved!")

Curriculum indentation corrected and file saved!


In [47]:
import py_compile

py_compile.compile(file_path, doraise=True)

print("SUCCESS: curriculum.py has valid Python syntax!")

SUCCESS: curriculum.py has valid Python syntax!


In [48]:
import sys
import importlib

member3_path = "/content/drive/MyDrive/LearnSyncAI/Member_3"

if member3_path not in sys.path:
    sys.path.insert(0, member3_path)

import curriculum
importlib.reload(curriculum)

test_mastery = {
    "arrays": 0.87,
    "linked_lists": 0.57
}

test_decision = curriculum.select_next_activity(
    "test_student",
    test_mastery,
    curriculum.DSA_PREREQUISITES,
    []
)

print("Curriculum decision:")
print(test_decision)

Curriculum decision:
{'student_id': 'test_student', 'concept_id': 'stacks', 'activity': 'reassessment', 'reason': 'Concept has not been assessed yet', 'next_action': 'reassess_mastery'}


In [49]:
required_fields = [
    "student_id",
    "concept_id",
    "activity",
    "reason",
    "next_action"
]

for field in required_fields:
    assert field in test_decision, f"Missing field: {field}"

print("SUCCESS: All five required curriculum fields are present!")

SUCCESS: All five required curriculum fields are present!


In [50]:
import os

base = "/content/drive/MyDrive/LearnSyncAI"

for root, dirs, files in os.walk(base):
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    if "rag.py" in files or "evaluator.py" in files:
        print("Folder:", root)
        print("Files:", files)

Folder: /content/drive/MyDrive/LearnSyncAI/Member_2
Files: ['opendsa_chunks.csv', 'short_answer_dataset.csv', 'rag.py', 'test_rag_pipeline.py', 'test_results.txt', 'heldout_answers.csv', 'evaluator.py.bak', 'test_member2.py', 'requirements.txt', 'run_heldout_eval.py.bak', 'test_evaluator.py.bak', 'final_test_results.txt', 'rag_development.ipynb.bak', 'evaluator_output.txt', 'rag_output.txt', 'member2_output.txt', 'heldout_output.txt', 'verified_full_run.txt', 'rag_evaluation_report.md.bak', 'run_heldout_eval.py', 'test_evaluator.py', 'evaluator.py.bak2', 'rag_development.ipynb', 'rag_evaluation_report.md.bak2', 'rag_evaluation_report.md.bak3', 'rag_evaluation_report.md.bak4', 'rag_evaluation_report.md.bak5', 'evaluator.py.bak3', 'rag_evaluation_report.md.bak6', 'rag_evaluation_report.md', 'evaluator.py']


In [51]:
import sys

member2_path = "/content/drive/MyDrive/LearnSyncAI/Member_2"

if member2_path not in sys.path:
    sys.path.insert(0, member2_path)

import rag
import evaluator

print("SUCCESS: Member 2 RAG module imported!")
print("SUCCESS: Member 2 evaluator module imported!")

SUCCESS: Member 2 RAG module imported!
SUCCESS: Member 2 evaluator module imported!


In [52]:
import rag
import evaluator

required_functions = {
    "RAG": [
        rag.retrieve_context,
        rag.generate_explanation
    ],
    "Evaluator": [
        evaluator.evaluate_answer,
        evaluator.bkt_outcome_from_evaluation
    ]
}

for module_name, functions in required_functions.items():
    for function in functions:
        print(f"SUCCESS: {module_name}.{function.__name__} is available!")

print("\nSUCCESS: All four Member 2 integration functions are available!")

SUCCESS: RAG.retrieve_context is available!
SUCCESS: RAG.generate_explanation is available!
SUCCESS: Evaluator.evaluate_answer is available!
SUCCESS: Evaluator.bkt_outcome_from_evaluation is available!

SUCCESS: All four Member 2 integration functions are available!


In [54]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 43.4 MB/s  0:00:00


In [55]:
import rag

test_concept = "arrays"
test_query = "What is an array in data structures?"

retrieved_context = rag.retrieve_context(
    query=test_query,
    concept_id=test_concept,
    top_k=3
)

print("Retrieved context:")
print(retrieved_context)

print("\nSUCCESS: RAG retrieval test completed!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieved context:
[{'chunk_id': 'chk_arr_01', 'concept_id': 'arrays', 'title': 'Array Data Structure and Random Access', 'source_url': 'https://opendsa-server.cs.vt.edu/ODSA/Books/CS3/html/ListArray.html', 'content': 'An array is a contiguous block of memory locations, each storing an element of the same data type. Elements are accessed using a zero-based integer index, providing constant-time O(1) random access via memory offset calculation (base_address + index * element_size). Inserting or deleting an element at an arbitrary position requires shifting all subsequent elements, resulting in O(n) worst-case time complexity. Fixed-size arrays cannot expand beyond their allocated capacity.', 'summary': 'Arrays store elements in contiguous memory enabling O(1) index access with O(n) arbitrary insertion or deletion.', 'difficulty_level': 'beginner', 'keywords': 'array, contiguous memory, indexing, random access, O(1), memory offset', 'score': 0.7057778239250183}, {'chunk_id': 'chk_arr_03'

In [56]:
explanation = rag.generate_explanation(
    query=test_query,
    retrieved_context=retrieved_context,
    mastery_level=0.5
)

print("Generated explanation:")
print(explanation)

print("\nSUCCESS: RAG explanation test completed!")

Generated explanation:
> *[Mode: Deterministic Offline Fallback (Reason: API key not configured)]*

## Instructional Response: What is an array in data structures?

**Pedagogical Target:** Intermediate Tier (Mastery: 50.0%)

### 🟡 Operational Mechanics & Implementation Details

**Topic:** Array Data Structure and Random Access

**Detailed Passage:**
An array is a contiguous block of memory locations, each storing an element of the same data type. Elements are accessed using a zero-based integer index, providing constant-time O(1) random access via memory offset calculation (base_address + index * element_size). Inserting or deleting an element at an arbitrary position requires shifting all subsequent elements, resulting in O(n) worst-case time complexity. Fixed-size arrays cannot expand beyond their allocated capacity.

**Operational Summary:** Arrays store elements in contiguous memory enabling O(1) index access with O(n) arbitrary insertion or deletion.

### 📚 Grounded OpenDSA Source

In [57]:
import evaluator
import pandas as pd
import os

dataset_path = os.path.join(
    member2_path,
    "short_answer_dataset.csv"
)

df = pd.read_csv(dataset_path)

print("Dataset columns:")
print(df.columns.tolist())

print("\nFirst available question:")
print(df.iloc[0].to_dict())

Dataset columns:
['question_id', 'concept_id', 'question_text', 'reference_answer', 'student_answer', 'ground_truth_classification', 'ground_truth_score', 'expected_misconceptions', 'data_source']

First available question:
{'question_id': 'Q_ARR_01', 'concept_id': 'arrays', 'question_text': 'Why do arrays provide O(1) random access, and what is the trade-off when inserting at the beginning?', 'reference_answer': 'Arrays store elements in contiguous memory locations, allowing direct memory address calculation (base + index * element_size) in O(1) time. Inserting at index 0 requires shifting all existing n elements right, taking O(n) time.', 'student_answer': 'Arrays use contiguous memory cells, so looking up any index is a direct math offset taking O(1) time. But inserting at the front requires moving every element one position to the right, which takes O(n) time.', 'ground_truth_classification': 'correct', 'ground_truth_score': 1.0, 'expected_misconceptions': nan, 'data_source': 'revi

In [58]:
row = df.iloc[0]

print("Question ID:", row["question_id"])
print("Concept:", row["concept_id"])
print("\nQuestion:")
print(row["question_text"])

print("\nReference answer:")
print(row["reference_answer"])

Question ID: Q_ARR_01
Concept: arrays

Question:
Why do arrays provide O(1) random access, and what is the trade-off when inserting at the beginning?

Reference answer:
Arrays store elements in contiguous memory locations, allowing direct memory address calculation (base + index * element_size) in O(1) time. Inserting at index 0 requires shifting all existing n elements right, taking O(n) time.


In [59]:
import evaluator

question_id = "Q_ARR_01"

student_answer = (
    "Arrays store elements in contiguous memory, so the computer "
    "can calculate an element's address directly using its index. "
    "This makes random access O(1). However, inserting at the "
    "beginning requires shifting existing elements, which takes O(n) time."
)

evaluation = evaluator.evaluate_answer(
    question_id=question_id,
    student_answer=student_answer
)

print("Evaluator result:")
print(evaluation)

print("\nBKT outcome:")
bkt_outcome = evaluator.bkt_outcome_from_evaluation(evaluation)
print(bkt_outcome)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluator result:
{'question_id': 'Q_ARR_01', 'concept_id': 'arrays', 'classification': 'correct', 'score': 1.0, 'bkt_outcome': 1, 'identified_misconceptions': [], 'feedback': 'Excellent! Your answer accurately captures the core principles and operational mechanics of arrays.', 'evidence': ['Supporting evidence (cosine similarity to reference): 0.8650', 'All required conceptual invariants verified.']}

BKT outcome:
1


In [60]:
import bkt_engine

student_id = "integration_test_student"

bkt_result = bkt_engine.update_bkt(
    student_id=student_id,
    concept_id=evaluation["concept_id"],
    outcome=bkt_outcome
)

print("BKT update result:")
print(bkt_result)

mastery_profile = bkt_engine.get_mastery_profile(student_id)

print("\nUpdated mastery profile:")
print(mastery_profile)

BKT update result:
{'student_id': 'integration_test_student', 'concept_id': 'arrays', 'outcome': 1, 'prior_mastery': 0.2, 'predicted_correct_probability': 0.3400000000000001, 'posterior_mastery': 0.5294117647058822, 'mastery': 0.5764705882352941}

Updated mastery profile:
{'student_id': 'integration_test_student', 'mastery': {'arrays': 0.5764705882352941}}


In [61]:
import sys

member3_path = "/content/drive/MyDrive/LearnSyncAI/Member_3"

if member3_path not in sys.path:
    sys.path.insert(0, member3_path)

import database

try:
    with database.get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT version();")
            result = cur.fetchone()

    print("SUCCESS: PostgreSQL connection established!")
    print("Database version:", result[0])

except Exception as e:
    print("DATABASE CONNECTION ERROR:")
    print(type(e).__name__, str(e))

SUCCESS: PostgreSQL connection established!
Database version: PostgreSQL 18.7 (a21afb5) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


In [62]:
import database

test_student_id = "integration_test_student"

try:
    database.create_student(test_student_id)

    print("SUCCESS: Test student created!")
    print("Student ID:", test_student_id)

except Exception as e:
    print("DATABASE ERROR:", type(e).__name__, str(e))

SUCCESS: Test student created!
Student ID: integration_test_student


In [63]:
# Convert Member 2's evaluation into Member 3's database format
attempt_evaluation = {
    **evaluation,
    "outcome": bkt_outcome
}

try:
    database.save_attempt(
        student_id=test_student_id,
        question_id=evaluation["question_id"],
        concept_id=evaluation["concept_id"],
        evaluation=attempt_evaluation
    )

    print("SUCCESS: Student attempt saved to PostgreSQL!")

except Exception as e:
    print("DATABASE ERROR:", type(e).__name__, str(e))

SUCCESS: Student attempt saved to PostgreSQL!


In [64]:
# Save the latest BKT mastery profile to PostgreSQL

try:
    database.save_mastery(
        student_id=test_student_id,
        mastery_profile=mastery_profile["mastery"]
    )

    print("SUCCESS: Mastery profile saved to PostgreSQL!")
    print("Student:", test_student_id)
    print("Mastery:", mastery_profile["mastery"])

except Exception as e:
    print("DATABASE ERROR:", type(e).__name__, str(e))

SUCCESS: Mastery profile saved to PostgreSQL!
Student: integration_test_student
Mastery: {'arrays': 0.5764705882352941}


In [65]:
history = database.get_student_history(test_student_id)

print("Retrieved student history:")
print(history)

print("\nNumber of recorded attempts:", len(history))

Retrieved student history:
[]

Number of recorded attempts: 0


In [66]:
with database.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT student_id, question_id, concept_id,
                   classification, outcome
            FROM attempts
            WHERE student_id = %s
            ORDER BY attempted_at DESC
        """, (test_student_id,))

        saved_attempts = cur.fetchall()

print("Saved attempts:")
for attempt in saved_attempts:
    print(attempt)

print("\nNumber of saved attempts:", len(saved_attempts))

Saved attempts:
('integration_test_student', 'Q_ARR_01', 'arrays', 'correct', 1)

Number of saved attempts: 1


In [67]:
with database.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT student_id, concept_id, mastery
            FROM mastery
            WHERE student_id = %s
        """, (test_student_id,))

        saved_mastery = cur.fetchall()

print("Saved mastery records:")
for record in saved_mastery:
    print(record)

print("\nNumber of mastery records:", len(saved_mastery))

Saved mastery records:
('integration_test_student', 'arrays', 0.5764705882352941)

Number of mastery records: 1


In [68]:
import curriculum

next_activity = curriculum.select_next_activity(
    student_id=test_student_id,
    mastery_profile=mastery_profile,
    prerequisites=curriculum.DSA_PREREQUISITES,
    learning_history=history
)

print("Selected next learning activity:")
print(next_activity)

Selected next learning activity:
{'student_id': 'integration_test_student', 'concept_id': 'linked_lists', 'activity': 'reassessment', 'reason': 'Concept has not been assessed yet. Recommended prior concepts with lower mastery: arrays. Learning is not blocked.', 'next_action': 'reassess_mastery'}


In [69]:
required_fields = [
    "student_id",
    "concept_id",
    "activity",
    "reason",
    "next_action"
]

print("Checking curriculum output fields:")

for field in required_fields:
    print(field, ":", field in next_activity)

assert all(field in next_activity for field in required_fields)

print("\nSUCCESS: All five required fields are present!")

Checking curriculum output fields:
student_id : True
concept_id : True
activity : True
reason : True
next_action : True

SUCCESS: All five required fields are present!


In [70]:
import inspect
import database

print(inspect.getsource(database.get_student_history))

def get_student_history(student_id):
    """
    Retrieve a student's learning history.

    Args:
        student_id (str): Student identifier.

    Returns:
        list[dict]: Learning history records ordered
                    from newest to oldest.

        Returns an empty list if no records exist.

    Raises:
        ValueError: If student_id is invalid.
        psycopg2.Error: If the database query fails.
    """

    _validate_required_string(student_id, "student_id")

    query = """
        SELECT
            history_id,
            student_id,
            concept_id,
            activity,
            reason,
            next_action,
            created_at

        FROM learning_history

        WHERE student_id = %s

        ORDER BY created_at DESC;
    """

    with _managed_connection() as conn:
        with conn.cursor(
            cursor_factory=RealDictCursor
        ) as cur:

            cur.execute(query, (student_id,))
            history = cur.fetchall()

    r

In [71]:
database.save_learning_history(
    student_id=test_student_id,
    concept_id=next_activity["concept_id"],
    activity=next_activity["activity"],
    reason=next_activity["reason"],
    next_action=next_activity["next_action"]
)

print("SUCCESS: Curriculum decision saved!")

SUCCESS: Curriculum decision saved!


In [72]:
history = database.get_student_history(test_student_id)

print("Retrieved learning history:")
print(history)

print("\nNumber of learning history records:", len(history))

Retrieved learning history:
[{'history_id': 6, 'student_id': 'integration_test_student', 'concept_id': 'linked_lists', 'activity': 'reassessment', 'reason': 'Concept has not been assessed yet. Recommended prior concepts with lower mastery: arrays. Learning is not blocked.', 'next_action': 'reassess_mastery', 'created_at': datetime.datetime(2026, 11, 18, 11, 53, 7, 33017, tzinfo=datetime.timezone.utc)}]

Number of learning history records: 1


In [73]:
import inspect
import orchestrator

print(inspect.getsource(orchestrator))


import database
import curriculum
import mock_agents


def run_learning_cycle(student_id, mastery_profile, prerequisites=None):
    """Select the next learning activity for a student."""

    if not student_id:
        raise ValueError("student_id is required")

    if not isinstance(mastery_profile, dict):
        raise TypeError("mastery_profile must be a dictionary")

    database.create_student(student_id)

    learning_history = database.get_student_history(student_id)

    if prerequisites is None:
        prerequisites = curriculum.DSA_PREREQUISITES

    decision = curriculum.select_next_activity(
        student_id,
        mastery_profile,
        prerequisites,
        learning_history
    )

    return {
        "student_id": student_id,
        "learning_history": learning_history,
        "decision": decision
    }


def get_next_question(student_id, mastery_profile, prerequisites=None):
    """Select a learning activity and generate a question."""

    cycle = run_learni

In [74]:
import assessment

questions = assessment.get_assessment_questions(
    concept_id="arrays"
)

print("Member 1 assessment question:")
print(questions[0])

Member 1 assessment question:
{'question_id': 'assess_arrays_01', 'concept_id': 'arrays', 'question': 'What is the time complexity of accessing an element using its index in an array?', 'options': {'A': 'O(1)', 'B': 'O(n)', 'C': 'O(log n)', 'D': 'O(n^2)'}}


In [75]:
import inspect
import assessment

print(inspect.getsource(
    assessment.evaluate_assessment_answer
))

def evaluate_assessment_answer(question_id, answer):
    """
    Evaluate an MCQ response.
    Accepts an option letter (A/B/C/D) or the exact option text.
    Returns a binary outcome suitable for BKT.
    """
    question = next(
        (q for q in QUESTION_BANK if q["question_id"] == question_id),
        None
    )

    if question is None:
        raise ValueError(f"Unknown question_id: {question_id}")

    if answer is None or not str(answer).strip():
        raise ValueError("Answer cannot be empty.")

    submitted = str(answer).strip().casefold()

    correct_option = question["correct_answer"]
    correct_text = question["options"][correct_option]

    valid_answers = {
        correct_option.casefold(),
        correct_text.casefold()
    }

    outcome = int(submitted in valid_answers)

    return {
        "question_id": question_id,
        "concept_id": question["concept_id"],
        "outcome": outcome,
        "classification": "correct" if outcome == 1 else "incorrec

In [76]:
import assessment
import bkt_engine

mcq_test_student = "integration_mcq_test_student"

# Get the arrays question
question = assessment.get_assessment_questions(
    concept_id="arrays"
)[0]

# Submit the correct answer for integration testing
mcq_evaluation = assessment.evaluate_assessment_answer(
    question_id=question["question_id"],
    answer=question["correct_answer"]
)

print("Member 1 evaluation:")
print(mcq_evaluation)

# Pass Member 1's outcome to Member 3 BKT
mcq_bkt_result = bkt_engine.update_bkt(
    student_id=mcq_test_student,
    concept_id=mcq_evaluation["concept_id"],
    outcome=mcq_evaluation["outcome"]
)

print("\nMember 3 BKT result:")
print(mcq_bkt_result)

print("\nSUCCESS: Member 1 → Member 3 integration works!")

KeyError: 'correct_answer'

In [77]:
question = assessment.get_assessment_questions(
    concept_id="arrays"
)[0]

print("Available question fields:")
print(question.keys())

print("\nComplete question:")
for key, value in question.items():
    print(f"{key}: {value}")

Available question fields:
dict_keys(['question_id', 'concept_id', 'question', 'options'])

Complete question:
question_id: assess_arrays_01
concept_id: arrays
question: What is the time complexity of accessing an element using its index in an array?
options: {'A': 'O(1)', 'B': 'O(n)', 'C': 'O(log n)', 'D': 'O(n^2)'}


In [78]:
# Evaluate the correct MCQ answer
mcq_evaluation = assessment.evaluate_assessment_answer(
    question_id=question["question_id"],
    answer="A"
)

print("Member 1 evaluation:")
print(mcq_evaluation)

# Update Member 3's BKT mastery
mcq_bkt_result = bkt_engine.update_bkt(
    student_id="integration_mcq_test_student",
    concept_id=mcq_evaluation["concept_id"],
    outcome=mcq_evaluation["outcome"]
)

print("\nMember 3 BKT result:")
print(mcq_bkt_result)

print("\nSUCCESS: Member 1 → Member 3 integration works!")

Member 1 evaluation:
{'question_id': 'assess_arrays_01', 'concept_id': 'arrays', 'outcome': 1, 'classification': 'correct', 'feedback': 'Correct! Array elements can be accessed directly using their index.'}

Member 3 BKT result:
{'student_id': 'integration_mcq_test_student', 'concept_id': 'arrays', 'outcome': 1, 'prior_mastery': 0.2, 'predicted_correct_probability': 0.3400000000000001, 'posterior_mastery': 0.5294117647058822, 'mastery': 0.5764705882352941}

SUCCESS: Member 1 → Member 3 integration works!


In [79]:
# Save the integrated MCQ attempt to PostgreSQL

database.create_student("integration_mcq_test_student")

database.save_attempt(
    student_id="integration_mcq_test_student",
    question_id=mcq_evaluation["question_id"],
    concept_id=mcq_evaluation["concept_id"],
    evaluation=mcq_evaluation
)

# Save the updated mastery
mastery_profile = bkt_engine.get_mastery_profile(
    "integration_mcq_test_student"
)

database.save_mastery(
    student_id="integration_mcq_test_student",
    mastery_profile=mastery_profile["mastery"]
)

print("SUCCESS: MCQ attempt and BKT mastery saved to PostgreSQL!")

SUCCESS: MCQ attempt and BKT mastery saved to PostgreSQL!


In [80]:
# Verify the attempt was saved
history = database.get_student_history(
    "integration_mcq_test_student"
)

print("Saved learning history:")
print(history)

# Verify the saved mastery directly
with database.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT concept_id, mastery
            FROM mastery
            WHERE student_id = %s
        """, ("integration_mcq_test_student",))

        saved_mastery = cur.fetchall()

print("\nSaved mastery:")
print(saved_mastery)

# Verify the saved attempt directly
with database.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT question_id, concept_id, classification, outcome
            FROM attempts
            WHERE student_id = %s
            ORDER BY attempted_at DESC
            LIMIT 1
        """, ("integration_mcq_test_student",))

        saved_attempt = cur.fetchone()

print("\nSaved attempt:")
print(saved_attempt)

Saved learning history:
[]

Saved mastery:
[('arrays', 0.5764705882352941)]

Saved attempt:
('assess_arrays_01', 'arrays', 'correct', 1)


In [81]:
import os

base = "/content/drive/MyDrive/LearnSyncAI"

for root, dirs, files in os.walk(base):
    dirs[:] = [d for d in dirs if d != "__pycache__"]

    if "rag.py" in files or "evaluator.py" in files:
        print("Folder:", root)
        print("Files:", files)

Folder: /content/drive/MyDrive/LearnSyncAI/Member_2
Files: ['opendsa_chunks.csv', 'short_answer_dataset.csv', 'rag.py', 'test_rag_pipeline.py', 'test_results.txt', 'heldout_answers.csv', 'evaluator.py.bak', 'test_member2.py', 'requirements.txt', 'run_heldout_eval.py.bak', 'test_evaluator.py.bak', 'final_test_results.txt', 'rag_development.ipynb.bak', 'evaluator_output.txt', 'rag_output.txt', 'member2_output.txt', 'heldout_output.txt', 'verified_full_run.txt', 'rag_evaluation_report.md.bak', 'run_heldout_eval.py', 'test_evaluator.py', 'evaluator.py.bak2', 'rag_development.ipynb', 'rag_evaluation_report.md.bak2', 'rag_evaluation_report.md.bak3', 'rag_evaluation_report.md.bak4', 'rag_evaluation_report.md.bak5', 'evaluator.py.bak3', 'rag_evaluation_report.md.bak6', 'rag_evaluation_report.md', 'evaluator.py']


In [82]:
import sys

member2_path = "/content/drive/MyDrive/LearnSyncAI/Member_2"

if member2_path not in sys.path:
    sys.path.insert(0, member2_path)

import rag
import evaluator

print("SUCCESS: Member 2 RAG imported!")
print("SUCCESS: Member 2 Evaluator imported!")

print("\nRAG functions:")
print(hasattr(rag, "retrieve_context"))
print(hasattr(rag, "generate_explanation"))

print("\nEvaluator functions:")
print(hasattr(evaluator, "evaluate_answer"))
print(hasattr(evaluator, "bkt_outcome_from_evaluation"))

SUCCESS: Member 2 RAG imported!
SUCCESS: Member 2 Evaluator imported!

RAG functions:
True
True

Evaluator functions:
True
True


In [83]:
import pandas as pd
import os

dataset_path = os.path.join(
    member2_path,
    "short_answer_dataset.csv"
)

df = pd.read_csv(dataset_path)

print("Dataset loaded successfully!")
print("Number of questions:", len(df))

# Display one real question
row = df.iloc[0]

print("\nQuestion ID:", row["question_id"])
print("Concept:", row["concept_id"])
print("Question:", row["question_text"])

Dataset loaded successfully!
Number of questions: 117

Question ID: Q_ARR_01
Concept: arrays
Question: Why do arrays provide O(1) random access, and what is the trade-off when inserting at the beginning?


In [84]:
# Use the sample student answer from Member 2's dataset
sample_answer = row["student_answer"]

evaluation = evaluator.evaluate_answer(
    question_id=row["question_id"],
    student_answer=sample_answer
)

print("Member 2 Evaluation:")
print(evaluation)

# Convert the evaluation into a BKT outcome
bkt_outcome = evaluator.bkt_outcome_from_evaluation(
    evaluation
)

print("\nBKT Outcome:")
print(bkt_outcome)

Member 2 Evaluation:
{'question_id': 'Q_ARR_01', 'concept_id': 'arrays', 'classification': 'correct', 'score': 1.0, 'bkt_outcome': 1, 'identified_misconceptions': [], 'feedback': 'Excellent! Your answer accurately captures the core principles and operational mechanics of arrays.', 'evidence': ['Supporting evidence (cosine similarity to reference): 0.8044', 'All required conceptual invariants verified.']}

BKT Outcome:
1


In [ ]:
# Update Member 3 BKT using Member 2's evaluation

student_id = "integration_member2_student"

database.create_student(student_id)

bkt_result = bkt_engine.update_bkt(
    student_id=student_id,
    concept_id=evaluation["concept_id"],
    outcome=bkt_outcome
)

print("BKT update result:")
print(bkt_result)

# Save the attempt
attempt_evaluation = {
    **evaluation,
    "outcome": bkt_outcome
}

database.save_attempt(
    student_id=student_id,
    question_id=evaluation["question_id"],
    concept_id=evaluation["concept_id"],
    evaluation=attempt_evaluation
)

# Save mastery
profile = bkt_engine.get_mastery_profile(student_id)

database.save_mastery(
    student_id=student_id,
    mastery_profile=profile["mastery"]
)

print("\nSUCCESS: Member 2 → Member 3 BKT and database integration works!")

BKT update result:
{'student_id': 'integration_member2_student', 'concept_id': 'arrays', 'outcome': 1, 'prior_mastery': 0.2, 'predicted_correct_probability': 0.3400000000000001, 'posterior_mastery': 0.5294117647058822, 'mastery': 0.5764705882352941}
